In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import textwrap

# LOAD DATA
df = pd.read_csv("questionnaire_data_other_factors.csv")

def merge_groups(g):
    if g["n"].sum() == 0: return pd.Series({"association": 0, "n": 0})
    return pd.Series({
        "association": np.average(g["association"], weights=g["n"]),
        "n": g["n"].sum()
    })

# Aggregate visits if multiple columns were merged
num_bin = df[df["type"].isin(["numeric", "binary"])].copy()
num_bin = num_bin.groupby(["score", "factor", "type"], as_index=False).apply(merge_groups, include_groups=False).reset_index()

cat = df[df["type"] == "categorical"].copy()
cat = cat.groupby(["score", "factor", "value"], as_index=False).apply(merge_groups, include_groups=False).reset_index()
cat["type"] = "categorical"

df = pd.concat([num_bin, cat], ignore_index=True)

OUT_DIR = Path("questionnaire_diagrams")
OUT_DIR.mkdir(exist_ok=True)

for questionnaire in sorted(df["score"].unique()):
    print(f"Processing {questionnaire}")
    qdf = df[df["score"] == questionnaire]
    
    for ftype in ["numeric", "binary", "categorical"]:
        subset = qdf[qdf["type"] == ftype].copy()
        if subset.empty: continue
        
        if ftype == "categorical":
            subset["label"] = [textwrap.fill(f"{row['factor']} = {row['value']}", width=40) for _, row in subset.iterrows()]
            subset = subset.sort_values(["factor", "association"])
        else:
            subset = subset.sort_values("association")
            subset["label"] = [textwrap.fill(str(l), width=40) for l in subset["factor"]]
            
        # Color by factor group
        unique_factors = subset["factor"].unique()
        colors_cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']
        color_map = {f: colors_cycle[i % len(colors_cycle)] for i, f in enumerate(unique_factors)}
        colors = [color_map[f] for f in subset["factor"]]
        
        plt.figure(figsize=(14, max(8, len(subset) * 0.6)))
        bars = plt.barh(subset["label"], subset["association"], color=colors, alpha=0.7)
        plt.axvline(0, color='black', linewidth=1)
        
        for i, bar in enumerate(bars):
            val = subset["association"].iloc[i]
            n = int(subset["n"].iloc[i])
            
            # Improved text placement to avoid overlapping axis
            if val >= 0:
                ha = 'left'
                x_pos = val + (subset["association"].max() * 0.01)
            else:
                ha = 'right'
                x_pos = val - (subset["association"].max() * 0.01)
                
            plt.text(x_pos, bar.get_y() + bar.get_height()/2, f'{val:.3f} (n={n})', 
                     va='center', ha=ha, fontsize=10, fontweight='bold' if abs(val) > 0.2 else 'normal')
            
        plt.title(f"{questionnaire.replace('_', ' ').title()} - {ftype.capitalize()} Factors", fontsize=18, pad=20)
        plt.xlabel("Spearman Correlation" if ftype == "numeric" else "Association (Mean Diff from global avg)", fontsize=14)
        plt.grid(axis='x', alpha=0.2)
        plt.tight_layout()
        plt.savefig(OUT_DIR / f"{questionnaire}_{ftype}.png", dpi=300)
        plt.close()

print(f"\nDONE. Saved to: {OUT_DIR}")

Processing chiq_result
Processing dass_depression
Processing dass_fear
Processing dass_stress
Processing gvas_result
Processing midas_result
Processing pgic_result

DONE. Saved to: questionnaire_diagrams
